# 01 - From data to a topo4d STAC catalog

In this notebook, you will learn how to transform raw multi-temporal geospatial data into a structured STAC catalog. Starting from point cloud epochs and satellite imagery, we will extract essential metadata, organize the data into STAC Items, and assemble them into a thematic Collection and Catalog.

This step is fundamental for enabling FAIR data management, and downstream analysis using spatiotemporal datasets where handling the temporal dimension is crucial.

By the end of this notebook, you will have created a minimal but valid STAC catalog that can be explored, shared, and used in further processing workflows.

### Learning objectives

- Understand why STAC is needed for topographic 4D data
- Learn the core STAC structure: Catalog -> Collection -> Item -> Asset
- Learn how to:
  - Extract metadata from raw data
  - Build a minimal but valid STAC catalog
- Understand time dimension handling (with topo4d)

<!-- ### Input

- Local or remote dataset
  - 5 point cloud epochs (`.copc.laz`)
  <!-- - 2 satellite images -->
- Minimal helper functions
  - metadata extraction (bbox, datetime, etc.)
  - STAC item creation template -->

<!-- ### Output

- A STAC catalog:
  - ```
    isar_stac/
    ├── catalog.json
    ├── collection.json
    ├── items/
    │   ├── epoch01.json
    │   ├── epoch02.json
    │   └── ...
- Each item includes:
  - spatial extent
  - timestamp
  - asset links (point cloud / raster)
  - topo4d extension fields -->


### Workflow

1. Import necessary libraries;
2. Configure the data and output folders;
3. Explore local point cloud files and construct metadata table;
4. Extract metadata from files;
5. Create one STAC Item and Asset;
6. Build all STAC Items;
7. Bulid Collection and Catalog;
8. Validate and save the Catalog;
9. Next steps.

## Expected input and output

### Input folder

The tutorial expects a folder with point-cloud files and a metadata table:

```text
data/isar_demo/
├── Isar_E0_UPH_5cm.las
├── Isar_E1_UPH_5cm.las
├── Isar_E2_ULS_5cm.las
├── Isar_E3_ULS_5cm.las
├── Isar_E4_ULS_5cm.las
└── metadata.csv
```

The point-cloud files may use `.copc.laz`, `.laz`, or `.las`.

### Minimum `metadata.csv` columns

```text
id,datetime,timezone,data_type,acquisition_mode,sensor,spatial_resolution,positional_accuracy,asset_url
```

Optional columns:

```text
public_asset_href,reference_epoch_id,trafor_file
```

`public_asset_href` can be used after the data are published on Zenodo or mediaTUM.

### Output folder

```text
data/catalog/
├── catalog.json
└── isar-pointclouds/
    ├── collection.json
    ├── Isar_E0_UPH_5cm/
    │   └── Isar_E0_UPH_5cm.json
    ├── Isar_E1_UPH_5cm/
    │   └── Isar_E1_UPH_5cm.json
    └── ...
```

## STAC concepts used in this notebook

| STAC concept | Meaning in this tutorial |
|---|---|
| Catalog | The root index of the dataset |
| Collection | The full Isar multi-epoch point-cloud dataset |
| Item | One point-cloud epoch |
| Asset | The actual `.copc.laz`, `.laz`, or `.las` file |
| Extension | Extra metadata for point clouds, projection, timestamps, and topo4d |

The important mapping is:

```text
one point cloud file  →  one STAC Item
one local or public file storage address  →  one STAC Asset
all epochs together  →  one STAC Collection
all collections together  →  one STAC Catalog
```

Core STAC fields describe where and when the data were observed. Extension fields describe domain-specific details such as point count, native CRS, acquisition mode, and topo4d metadata.


## 1. Imports

This notebook keeps all helper and builder functions inside the notebook. No external helper module is required.

**Required packages:**

- `pandas` and `numpy` for tables and matrices;
- `laspy` for LAS/LAZ/COPC header metadata;
- `pyproj` and `shapely` for CRS handling and geometry creation;
- `pystac` for STAC objects;
- `topo4d` for the topo4d STAC extension.

Run this notebook from the repository root or from `docs/tutorial/` so that the local `topo4d` package can be imported.

In [ ]:
# installation
# !pip install pystac # uncomment and run this line if you didn't install it yet

In [1]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple
from zoneinfo import ZoneInfo

import json
import os
import sys

import numpy as np
import pandas as pd
import pystac
from dateutil import parser
from pyproj import CRS, Transformer
from shapely.geometry import box, mapping
import laspy


# If the notebook is run from docs/tutorial/, add the repository root to Python's path.
# This makes `import topo4d` work in a source checkout.
repo_root = Path.cwd().resolve().parents[1] if len(Path.cwd().resolve().parents) >= 2 else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

try:
    from topo4d.topo4d_ext import Topo4DExtension, DataType, TrafoMeta
except ImportError as exc:
    raise ImportError(
        "Could not import the topo4d extension. Install the package or run this notebook "
        "from a checkout where the topo4d package is importable."
    ) from exc

print("Imports successful.")

Imports successful.


## 2. Configuration

Set the input path, output path, asset-link mode, and basic STAC metadata here.

Use:

- `ASSET_MODE = "local"` while developing from local files;
- `ASSET_MODE = "public"` after the files are published and `metadata.csv` contains public asset URLs.


In [3]:
# data_root = "./data"
# data_dir = f"{data_root}/Isar_tutorial"

# Data input
DATA_ROOT = Path("./data/Isar_tutorial").resolve()
METADATA_CSV = DATA_ROOT / "metadata.csv"

# STAC output
CATALOG_ROOT = DATA_ROOT / "catalog"

# Asset link mode: "local" or "public"
ASSET_MODE = "local"

# STAC metadata
CATALOG_ID = "isar-demo-catalog"
CATALOG_DESCRIPTION = "A demo STAC Catalog using topo4d extension for the Isar multi-temporal point cloud tutorial."

COLLECTION_ID = "isar-pointclouds"
COLLECTION_TITLE = "Isar multi-temporal point clouds"
COLLECTION_DESCRIPTION = (
    "Five point cloud epochs from the Isar riverbed area, prepared for the topo4d STAC tutorial."
)
COLLECTION_LICENSE = "CC-BY-4.0"

print("DATA_ROOT:", DATA_ROOT.as_posix())
print("METADATA_CSV:", METADATA_CSV.as_posix())
print("CATALOG_ROOT:", CATALOG_ROOT.as_posix())
print("ASSET_MODE:", ASSET_MODE)

DATA_ROOT: C:/Users/jiapan/02_projects/2025_4DWORKS/ISPRS_tutorial/4D-WORKS-dev/docs/tutorial/data/Isar_tutorial
METADATA_CSV: C:/Users/jiapan/02_projects/2025_4DWORKS/ISPRS_tutorial/4D-WORKS-dev/docs/tutorial/data/Isar_tutorial/metadata.csv
CATALOG_ROOT: C:/Users/jiapan/02_projects/2025_4DWORKS/ISPRS_tutorial/4D-WORKS-dev/docs/tutorial/data/Isar_tutorial/catalog
ASSET_MODE: local


## 3. Helper functions: file discovery and metadata table

These functions answer two basic questions:

1. Which point cloud files are available?
2. Which user-provided metadata row belongs to each file?

The file ID is derived from the filename. For example:

```text
Isar_E0_UPH_5cm.las → Isar_E0_UPH_5cm
```


In [11]:
POINTCLOUD_SUFFIXES = (".copc.laz", ".laz", ".las")


def pointcloud_id(path: Path) -> str:
    """Return the dataset ID from a LAS/LAZ/COPC filename."""
    name = path.name
    lower_name = name.lower()

    for suffix in POINTCLOUD_SUFFIXES:
        if lower_name.endswith(suffix):
            return name[: -len(suffix)]

    return path.stem


def discover_pointclouds(data_root: Path) -> List[Path]:
    """Find LAS, LAZ, and COPC LAZ files in a folder."""
    files = [
        path
        for path in sorted(data_root.iterdir())
        if path.is_file() and path.name.lower().endswith(POINTCLOUD_SUFFIXES)
    ]

    if not files:
        raise FileNotFoundError(f"No point-cloud files found in {data_root}")

    return files


def load_metadata_table(csv_path: Path) -> pd.DataFrame:
    """Load metadata.csv and use the `id` column as the table index."""
    if not csv_path.exists():
        raise FileNotFoundError(f"Metadata table not found: {csv_path}")

    table = pd.read_csv(csv_path)

    if "id" not in table.columns:
        raise ValueError("metadata.csv must contain an 'id' column.")

    table = table.set_index("id", drop=False)
    return table


def metadata_for_file(file_path: Path, metadata_table: pd.DataFrame) -> Dict[str, Any]:
    """Return the metadata row that matches a point-cloud file."""
    item_id = pointcloud_id(file_path)

    if item_id not in metadata_table.index:
        raise KeyError(f"No metadata.csv row found for point-cloud ID: {item_id}")

    row = metadata_table.loc[item_id].to_dict()

    # Convert NaN values to None. This keeps later JSON output clean.
    return {key: (None if pd.isna(value) else value) for key, value in row.items()}

### Check the input data

Run the next cell before building the catalog. It confirms that every discovered point-cloud file has a matching row in `metadata.csv`.

In [12]:
pointcloud_files = discover_pointclouds(DATA_ROOT)
metadata_table = load_metadata_table(METADATA_CSV)

input_check = pd.DataFrame(
    {
        "file": [path.name for path in pointcloud_files],
        "item_id": [pointcloud_id(path) for path in pointcloud_files],
        "has_metadata_row": [pointcloud_id(path) in metadata_table.index for path in pointcloud_files],
    }
)

input_check

,file,item_id,has_metadata_row
0,Isar_E0_UPH_5cm.las,Isar_E0_UPH_5cm,True
1,Isar_E1_UPH_5cm.las,Isar_E1_UPH_5cm,True
2,Isar_E2_ULS_5cm.las,Isar_E2_ULS_5cm,True
3,Isar_E3_ULS_5cm.las,Isar_E3_ULS_5cm,True
4,Isar_E4_ULS_5cm.las,Isar_E4_ULS_5cm,True


## 4. Helper functions: datetime, geometry, and LAS/LAZ metadata

A STAC Item needs:

- a stable ID;
- a timezone-aware observation datetime;
- a geometry and bounding box in WGS84;
- point-cloud metadata such as point count and native CRS.

The following helpers extract these values from `metadata.csv` and the point-cloud header.


In [13]:
def parse_datetime_utc(row: Dict[str, Any]) -> datetime:
    """Parse the acquisition datetime and return it in UTC."""
    if not row.get("datetime"):
        raise ValueError("Missing required metadata field: datetime")

    dt = parser.parse(str(row["datetime"]))

    timezone_value = row.get("timezone")

    if dt.tzinfo is None:
        if timezone_value and str(timezone_value).startswith("UTC"):
            # Examples: UTC+01:00, UTC+02:00, UTC-05:30
            tz_text = str(timezone_value).replace("UTC", "")
            sign = -1 if tz_text.startswith("-") else 1
            tz_text = tz_text.replace("+", "").replace("-", "")
            hours, minutes = (tz_text.split(":") + ["0"])[:2]
            offset = timedelta(hours=sign * int(hours or 0), minutes=sign * int(minutes or 0))
            dt = dt.replace(tzinfo=timezone(offset))
        elif timezone_value:
            dt = dt.replace(tzinfo=ZoneInfo(str(timezone_value)))
        else:
            dt = dt.replace(tzinfo=timezone.utc)

    return dt.astimezone(timezone.utc)


def read_pointcloud_header(file_path: Path) -> Dict[str, Any]:
    """Read core metadata from a LAS/LAZ/COPC header with laspy."""
    with laspy.open(file_path) as pointcloud:
        header = pointcloud.header

        try:
            native_crs = header.parse_crs()
        except Exception:
            native_crs = None

        return {
            "point_count": int(header.point_count),
            "mins": [float(value) for value in header.mins],
            "maxs": [float(value) for value in header.maxs],
            "native_crs": native_crs,
        }


def crs_to_proj_code(crs: Optional[CRS]) -> Optional[str]:
    """Convert a CRS object to a compact proj:code value, when possible."""
    if crs is None:
        return None

    authority = crs.to_authority()
    if authority:
        return f"{authority[0]}:{authority[1]}"

    return crs.to_string()


def geometry_from_header(header: Dict[str, Any]) -> Tuple[Dict[str, Any], List[float], Optional[str]]:
    """Create WGS84 geometry, WGS84 bbox, and proj:code from header metadata."""
    xmin, ymin, _ = header["mins"]
    xmax, ymax, _ = header["maxs"]

    native_crs = header.get("native_crs")
    proj_code = crs_to_proj_code(native_crs)

    if native_crs and native_crs.to_epsg() != 4326:
        transformer = Transformer.from_crs(native_crs, CRS.from_epsg(4326), always_xy=True)
        corners = [
            transformer.transform(xmin, ymin),
            transformer.transform(xmin, ymax),
            transformer.transform(xmax, ymin),
            transformer.transform(xmax, ymax),
        ]
        xs, ys = zip(*corners)
        bbox_wgs84 = [min(xs), min(ys), max(xs), max(ys)]
    else:
        bbox_wgs84 = [xmin, ymin, xmax, ymax]

    geometry = mapping(box(*bbox_wgs84))
    return geometry, bbox_wgs84, proj_code


def pointcloud_media_type(file_path: Path) -> str:
    """Return a STAC media type for the point-cloud asset."""
    name = file_path.name.lower()

    if name.endswith(".copc.laz"):
        return "application/vnd.laszip+copc"
    if name.endswith(".laz"):
        return "application/vnd.laszip"
    if name.endswith(".las"):
        return "application/vnd.las"

    return "application/octet-stream" 

## 5. Builder functions: Asset and Item

The next two functions are the core builders.

They are intentionally direct:

- `build_asset(...)` creates one STAC Asset from one file path.
- `build_item(...)` creates one STAC Item from one point-cloud file and one metadata row.

This pattern is easy to test, teach, and reuse.


In [14]:
STAC_EXTENSIONS = [
    "https://stac-extensions.github.io/projection/v2.0.0/schema.json",
    "https://stac-extensions.github.io/pointcloud/v2.0.0/schema.json",
    "https://stac-extensions.github.io/timestamps/v1.1.0/schema.json",
]


def asset_href(file_path: Path, row: Dict[str, Any]) -> str:
    """Choose the asset HREF for local or public catalog mode."""
    if ASSET_MODE == "public":
        href = row.get("public_asset_href") or row.get("asset_url")
        if not href:
            raise ValueError(
                f"ASSET_MODE='public' requires public_asset_href or asset_url for {row.get('id')}"
            )
        return str(href)

    return file_path.as_posix()


def build_asset(file_path: Path, row: Dict[str, Any]) -> Tuple[str, pystac.Asset]:
    """Build the STAC Asset for one point-cloud file."""
    asset_key = row.get("data_type") or "pointcloud"

    asset = pystac.Asset(
        href=asset_href(file_path, row),
        media_type=pointcloud_media_type(file_path),
        roles=["data"],
        title=f"Point cloud asset for {row.get('id', pointcloud_id(file_path))}",
    )

    return asset_key, asset


def build_item(file_path: Path, row: Dict[str, Any]) -> pystac.Item:
    """Build one STAC Item from one point-cloud file and one metadata row."""
    item_id = row.get("id") or pointcloud_id(file_path)
    header = read_pointcloud_header(file_path)
    geometry, bbox, proj_code = geometry_from_header(header)
    item_datetime = parse_datetime_utc(row)

    item = pystac.Item(
        id=item_id,
        geometry=geometry,
        bbox=bbox,
        datetime=item_datetime,
        properties={},
    )

    item.stac_extensions.extend(STAC_EXTENSIONS)

    # Standard STAC extension fields
    item.properties["proj:code"] = proj_code
    item.properties["pc:count"] = header["point_count"]
    item.properties["pc:type"] = "lidar" if row.get("acquisition_mode") in {"ALS", "ULS", "TLS"} else "other"

    if row.get("sensor"):
        item.properties["instruments"] = [row["sensor"]]

    # topo4d extension fields
    topo4d = Topo4DExtension.ext(item, add_if_missing=True)

    # The extension has used both names during development; support the installed one.
    if hasattr(topo4d, "timezone"):
        topo4d.timezone = row.get("timezone") or "UTC"
    elif hasattr(topo4d, "tz"):
        topo4d.tz = row.get("timezone") or "UTC"

    topo4d.data_type = DataType(row.get("data_type") or "pointcloud")
    topo4d.acquisition_mode = row.get("acquisition_mode")
    topo4d.orientation = row.get("orientation")
    topo4d.spatial_resolution = row.get("spatial_resolution")
    topo4d.positional_accuracy = row.get("positional_accuracy")

    asset_key, asset = build_asset(file_path, row)
    item.add_asset(asset_key, asset)

    return item

### Build and inspect one Item

Before looping over every file, build one Item and inspect the result. This is the most useful debugging step in the notebook.


In [15]:
example_file = pointcloud_files[0]
example_row = metadata_for_file(example_file, metadata_table)

example_item = build_item(example_file, example_row)

print("Item ID:", example_item.id)
print("Datetime:", example_item.datetime)
print("BBox:", example_item.bbox)
print("Assets:", list(example_item.assets.keys()))

example_item.to_dict()

Item ID: Isar_E0_UPH_5cm
Datetime: 2024-08-12 13:26:52+00:00
BBox: [11.311183042859497, 47.52983725859918, 11.313525437239518, 47.531227393971676]
Assets: ['pointcloud']


{'type': 'Feature',
 'stac_version': '1.1.0',
 'stac_extensions': ['https://stac-extensions.github.io/projection/v2.0.0/schema.json',
  'https://stac-extensions.github.io/pointcloud/v2.0.0/schema.json',
  'https://stac-extensions.github.io/timestamps/v1.1.0/schema.json',
  'https://tum-rsa.github.io/topo4d/v0.2.0/schema.json'],
 'id': 'Isar_E0_UPH_5cm',
 'geometry': {'type': 'Polygon',
  'coordinates': (((11.313525437239518, 47.52983725859918),
    (11.313525437239518, 47.531227393971676),
    (11.311183042859497, 47.531227393971676),
    (11.311183042859497, 47.52983725859918),
    (11.313525437239518, 47.52983725859918)),)},
 'bbox': [11.311183042859497,
  47.52983725859918,
  11.313525437239518,
  47.531227393971676],
 'properties': {'proj:code': 'EPSG:32632',
  'pc:count': 2201140,
  'pc:type': 'other',
  'instruments': ['DJI Phantom 4 RTK'],
  'topo4d:timezone': 'UTC+1',
  'topo4d:data_type': 'pointcloud',
  'topo4d:acquisition_mode': 'UPH',
  'topo4d:orientation': 'Nadir',
  'top

## 6. Build all Items

Now apply the same builder to every discovered point-cloud file.


In [16]:
items = []

for file_path in pointcloud_files:
    row = metadata_for_file(file_path, metadata_table)
    item = build_item(file_path, row)
    items.append(item)

items_by_id = {item.id: item for item in items}

item_summary = pd.DataFrame(
    {
        "item_id": [item.id for item in items],
        "datetime_utc": [item.datetime.isoformat() for item in items],
        "asset_key": [next(iter(item.assets.keys())) for item in items],
        "asset_href": [next(iter(item.assets.values())).href for item in items],
        "pc_count": [item.properties.get("pc:count") for item in items],
        "proj_code": [item.properties.get("proj:code") for item in items],
    }
)

item_summary

,item_id,datetime_utc,asset_key,asset_href,pc_count,proj_code
0,Isar_E0_UPH_5cm,2024-08-12T13:26:52+00:00,pointcloud,C:/Users/jiapan/02_projects/2025_4DWORKS/ISPRS...,2201140,EPSG:32632
1,Isar_E1_UPH_5cm,2024-11-05T10:04:34+00:00,pointcloud,C:/Users/jiapan/02_projects/2025_4DWORKS/ISPRS...,2680542,EPSG:32632
2,Isar_E2_ULS_5cm,2025-03-25T12:15:51+00:00,pointcloud,C:/Users/jiapan/02_projects/2025_4DWORKS/ISPRS...,3262299,EPSG:32632
3,Isar_E3_ULS_5cm,2025-09-09T09:14:44+00:00,pointcloud,C:/Users/jiapan/02_projects/2025_4DWORKS/ISPRS...,4568259,EPSG:32632
4,Isar_E4_ULS_5cm,2026-03-24T09:28:58+00:00,pointcloud,C:/Users/jiapan/02_projects/2025_4DWORKS/ISPRS...,5084530,EPSG:32632


## Optional topo4d transformation metadata

If `metadata.csv` contains `reference_epoch_id` and `transformation_file`, this section attaches a transformation matrix to each relevant Item.

Expected columns:

```text
id,reference_epoch_id,transformation_file
```

Example:

```text
Isar_E2_UPH_5cm,Isar_E1_UPH_5cm,trafo_E2_to_E1.txt
```

Rows without a transformation file are skipped.


In [17]:
def read_transformation_matrix(path: Path) -> List[List[float]]:
    """Read a whitespace-separated transformation matrix from a text file."""
    matrix = np.loadtxt(path)
    return matrix.tolist()


def add_transformation_metadata(
    item: pystac.Item,
    row: Dict[str, Any],
    items_by_id: Dict[str, pystac.Item],
) -> None:
    """Attach topo4d transformation metadata when metadata.csv provides it."""
    transformation_file = row.get("transformation_file")
    reference_epoch_id = row.get("reference_epoch_id")

    if not transformation_file:
        return

    if not reference_epoch_id:
        raise ValueError(f"{item.id} has transformation_file but no reference_epoch_id.")

    if reference_epoch_id not in items_by_id:
        raise ValueError(f"Reference epoch not found: {reference_epoch_id}")

    transformation_path = DATA_ROOT / str(transformation_file)
    if not transformation_path.exists():
        raise FileNotFoundError(f"Transformation file not found: {transformation_path}")

    reference_item = items_by_id[reference_epoch_id]
    reference_link = pystac.Link(
        rel="reference",
        target=reference_item,
        title=f"Reference epoch: {reference_item.id}",
        media_type=pystac.MediaType.JSON,
    )

    trafo_meta = TrafoMeta.create(
        reference_epoch=reference_link.to_dict(),
        transformation=read_transformation_matrix(transformation_path),
    )

    Topo4DExtension.ext(item, add_if_missing=True).trafometa = trafo_meta


if {"reference_epoch_id", "transformation_file"}.issubset(metadata_table.columns):
    for file_path in pointcloud_files:
        row = metadata_for_file(file_path, metadata_table)
        add_transformation_metadata(items_by_id[row["id"]], row, items_by_id)

    print("Transformation metadata checked.")
else:
    print("No transformation metadata columns found. Skipping this optional section.")

No transformation metadata columns found. Skipping this optional section.


## 7. Builder functions: Collection and Catalog

A Collection describes the dataset as a whole.  
A Catalog is the root entry point that links to the Collection.

These two builders are short because most of the detailed metadata already lives in the Items.


In [18]:
def build_collection(items: Iterable[pystac.Item]) -> pystac.Collection:
    """Build one STAC Collection from the point-cloud Items."""
    items = list(items)

    if not items:
        raise ValueError("Cannot build a Collection without Items.")

    collection = pystac.Collection(
        id=COLLECTION_ID,
        title=COLLECTION_TITLE,
        description=COLLECTION_DESCRIPTION,
        license=COLLECTION_LICENSE,
        extent=pystac.Extent.from_items(items),
        keywords=["topo4d", "point cloud", "4D", "STAC", "Isar"],
        providers=[
            pystac.Provider(
                name="Technical University of Munich",
                roles=["producer", "processor"],
                url="https://www.tum.de/",
            )
        ],
    )

    acquisition_modes = sorted(
        {item.properties.get("topo4d:acquisition_mode") for item in items if item.properties.get("topo4d:acquisition_mode")}
    )
    data_types = sorted(
        {item.properties.get("topo4d:data_type") for item in items if item.properties.get("topo4d:data_type")}
    )

    if acquisition_modes:
        collection.summaries.add("topo4d:acquisition_mode", acquisition_modes)
    if data_types:
        collection.summaries.add("topo4d:data_type", data_types)

    for item in items:
        collection.add_item(item)

    return collection


def build_catalog(collection: pystac.Collection) -> pystac.Catalog:
    """Build the root STAC Catalog and attach the Collection."""
    catalog = pystac.Catalog(
        id=CATALOG_ID,
        description=CATALOG_DESCRIPTION,
        title="Isar topo4d STAC Catalog",
    )

    catalog.add_child(collection)
    return catalog


collection = build_collection(items)
catalog = build_catalog(collection)

catalog.describe()

* <Catalog id=isar-demo-catalog>
    * <Collection id=isar-pointclouds>
      * <Item id=Isar_E0_UPH_5cm>
      * <Item id=Isar_E1_UPH_5cm>
      * <Item id=Isar_E2_ULS_5cm>
      * <Item id=Isar_E3_ULS_5cm>
      * <Item id=Isar_E4_ULS_5cm>


## 8. Save and validate the catalog

The catalog is saved as a self-contained STAC catalog with relative links between STAC JSON files.

Asset HREFs remain controlled by `ASSET_MODE`:

- `local`: assets point to local files;
- `public`: assets point to URLs from `metadata.csv`.


In [22]:
CATALOG_ROOT.mkdir(parents=True, exist_ok=True)

catalog.normalize_hrefs(CATALOG_ROOT.as_posix())
catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)

print(f"Catalog saved to: {CATALOG_ROOT / 'catalog.json'}")

Catalog saved to: C:\Users\jiapan\02_projects\2025_4DWORKS\ISPRS_tutorial\4D-WORKS-dev\docs\tutorial\data\Isar_Wallgau_part\catalog\catalog.json


In [20]:
# Validate the catalog structure and all linked STAC objects.
# Validation requires internet access if schemas are fetched remotely for the first time.
try:
    catalog.validate_all()
    print("STAC validation successful.")
except Exception as exc:
    print("Validation could not be completed in this environment.")
    print(type(exc).__name__ + ":", exc)

STAC validation successful.


### Inspect the final files

Use this check to confirm which STAC JSON files were written.

The expected output includes:

- one root `catalog.json`;
- one Collection JSON;
- one Item JSON per point-cloud epoch.

In [21]:
written_files = sorted(path.relative_to(CATALOG_ROOT) for path in CATALOG_ROOT.rglob("*.json"))
written_files

[WindowsPath('catalog.json'),
 WindowsPath('isar-pointclouds/collection.json'),
 WindowsPath('isar-pointclouds/Isar_E0_UPH_5cm/Isar_E0_UPH_5cm.json'),
 WindowsPath('isar-pointclouds/Isar_E1_UPH_5cm/Isar_E1_UPH_5cm.json'),
 WindowsPath('isar-pointclouds/Isar_E2_ULS_5cm/Isar_E2_ULS_5cm.json'),
 WindowsPath('isar-pointclouds/Isar_E3_ULS_5cm/Isar_E3_ULS_5cm.json'),
 WindowsPath('isar-pointclouds/Isar_E4_ULS_5cm/Isar_E4_ULS_5cm.json')]

## 9. Next steps


### Switching from local files to Zenodo or mediaTUM links

After publishing the point-cloud files, add a public URL column to `metadata.csv`:

```text
id,public_asset_href
Isar_E0_UPH_5cm,https://zenodo.org/records/.../files/Isar_E0_UPH_5cm.copc.laz
```

Then rerun the notebook with:

```python
ASSET_MODE = "public"
```

Only the asset HREFs change. The Item, Collection, and Catalog structure stays the same.

### The NB1 output is a STAC Catalog for the Isar point clouds.

Recommended follow-up notebooks:

1. **NB2 — Query Catalog**: filter Items by time, acquisition mode, or asset metadata and download selected assets.
2. **NB3 — Catalog for Analysis**: run a simple analysis such as M3C2 and visualize the result.
3. **Demo**: serve a static STAC Catalog and a Potree project for a separate dataset such as Kijkduin.

## References

- STAC specification: <https://stacspec.org/>
- PySTAC documentation: <https://pystac.readthedocs.io/>
- STAC Point Cloud Extension: <https://github.com/stac-extensions/pointcloud>
- STAC Projection Extension: <https://github.com/stac-extensions/projection>
- STAC Timestamps Extension: <https://github.com/stac-extensions/timestamps>
- Zenodo developer documentation: <https://developers.zenodo.org/>
- Zenodo deposit documentation: <https://help.zenodo.org/docs/deposit/>